# 10 – RAG-Powered Estimating Assistant (LLM + Retrieval)
## Business Context
When a technician needs to write an estimate, they look up line items in a **price book** —
a catalog of standard job codes with labor and material prices. This is:
- Time-consuming (sorting through hundreds of codes)
- Error-prone (wrong code = wrong price)
- Inconsistent (different techs pick different codes for the same work)

An LLM-powered assistant can:
1. Accept a natural language description of the work performed
2. Retrieve the most relevant price book entries (RAG)
3. Draft a structured estimate for technician review

## Architecture
```
Tech description (voice/text)
  → Embedding model → query vector
  → Vector search over price book embeddings → top-K candidates
  → LLM prompt: "Given these candidates, draft the estimate"
  → Structured JSON output → populate estimate form
  → Tech reviews and approves
```

This is a **Retrieval-Augmented Generation (RAG)** system.
The retrieval step is critical — without it, the LLM hallucinates price book codes.


In [1]:
import numpy as np
import pandas as pd
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)


In [2]:
# ─── Synthetic Price Book ─────────────────────────────────────────────────────
# In production this is loaded from ServiceTitan's price book database.
# Each entry has a code, description, unit labor cost, and unit parts cost.

PRICE_BOOK = [
    # HVAC Cooling
    {"code": "HVAC-001", "category": "cooling", "description": "AC system diagnostic and inspection, includes refrigerant level check", "labor": 75, "parts": 0},
    {"code": "HVAC-002", "category": "cooling", "description": "Refrigerant recharge R-410A, per pound", "labor": 20, "parts": 45},
    {"code": "HVAC-003", "category": "cooling", "description": "Condenser fan motor replacement, standard residential unit", "labor": 120, "parts": 180},
    {"code": "HVAC-004", "category": "cooling", "description": "Run capacitor replacement, dual run capacitor 35/5 MFD", "labor": 60, "parts": 45},
    {"code": "HVAC-005", "category": "cooling", "description": "Contactor replacement, 2-pole 30 amp", "labor": 60, "parts": 35},
    {"code": "HVAC-006", "category": "cooling", "description": "AC evaporator coil cleaning, indoor air handler unit", "labor": 95, "parts": 25},
    {"code": "HVAC-007", "category": "cooling", "description": "AC condensate drain line flush and clean, prevent blockage", "labor": 65, "parts": 10},
    {"code": "HVAC-008", "category": "cooling", "description": "Thermostat replacement, standard programmable thermostat installation", "labor": 75, "parts": 120},
    {"code": "HVAC-009", "category": "cooling", "description": "Blower motor replacement, indoor air handler", "labor": 150, "parts": 220},
    # HVAC Heating
    {"code": "HVAC-101", "category": "heating", "description": "Furnace diagnostic, includes heat exchanger inspection", "labor": 75, "parts": 0},
    {"code": "HVAC-102", "category": "heating", "description": "Igniter replacement, silicon nitride hot surface igniter", "labor": 80, "parts": 55},
    {"code": "HVAC-103", "category": "heating", "description": "Flame sensor cleaning and adjustment", "labor": 65, "parts": 5},
    {"code": "HVAC-104", "category": "heating", "description": "Draft inducer motor replacement, residential furnace", "labor": 140, "parts": 240},
    {"code": "HVAC-105", "category": "heating", "description": "Pressure switch replacement, furnace", "labor": 80, "parts": 55},
    {"code": "HVAC-106", "category": "heating", "description": "Gas valve replacement, residential furnace 24V", "labor": 160, "parts": 180},
    # Plumbing
    {"code": "PLMB-001", "category": "plumbing", "description": "Drain snaking, kitchen or bathroom sink, up to 50 feet", "labor": 120, "parts": 0},
    {"code": "PLMB-002", "category": "plumbing", "description": "Water heater replacement, 40 gallon gas standard", "labor": 280, "parts": 680},
    {"code": "PLMB-003", "category": "plumbing", "description": "Toilet repair, fill valve and flapper replacement", "labor": 85, "parts": 35},
    {"code": "PLMB-004", "category": "plumbing", "description": "Faucet replacement, single handle kitchen faucet, customer-supplied fixture", "labor": 120, "parts": 0},
    {"code": "PLMB-005", "category": "plumbing", "description": "Water heater flush and sediment removal, maintenance", "labor": 90, "parts": 15},
    # Electrical
    {"code": "ELEC-001", "category": "electrical", "description": "GFCI outlet replacement, bathroom or kitchen", "labor": 75, "parts": 25},
    {"code": "ELEC-002", "category": "electrical", "description": "Circuit breaker replacement, 15 or 20 amp single pole", "labor": 95, "parts": 35},
    {"code": "ELEC-003", "category": "electrical", "description": "Ceiling fan installation, wiring and mounting, existing box", "labor": 110, "parts": 0},
    {"code": "ELEC-004", "category": "electrical", "description": "Panel inspection and labeling, residential", "labor": 120, "parts": 10},
    # Service calls
    {"code": "SVC-001", "category": "service", "description": "Standard service call fee, weekday business hours", "labor": 89, "parts": 0},
    {"code": "SVC-002", "category": "service", "description": "After-hours emergency service call fee, evenings and weekends", "labor": 149, "parts": 0},
    {"code": "SVC-003", "category": "service", "description": "Annual maintenance plan, HVAC tune-up, 2 visits per year", "labor": 0, "parts": 0},
]

pb_df = pd.DataFrame(PRICE_BOOK)
pb_df['total_price'] = pb_df['labor'] + pb_df['parts']
print(f"Price book: {len(pb_df)} line items across {pb_df['category'].nunique()} categories")
print(pb_df.groupby('category')['total_price'].agg(['count','mean']).round(2))


Price book: 27 line items across 5 categories
            count    mean
category                 
cooling         9  155.56
electrical      4  117.50
heating         6  189.17
plumbing        5  285.00
service         3   79.33


## TF-IDF Retrieval (Lightweight RAG)
In production, you'd use dense embeddings (e.g., text-embedding-3-small + pgvector).
TF-IDF is a useful stand-in that makes the retrieval logic transparent and
works well for structured domains like price books (limited vocabulary).

The retrieval pipeline:
1. Embed all price book descriptions offline
2. At query time: embed the technician description, find top-K cosine-similar entries
3. Pass top-K entries to the LLM as context


In [3]:
# ─── Build TF-IDF Index over Price Book ──────────────────────────────────────
# In production: replace with OpenAI text-embedding-3-small + FAISS/pgvector

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, stop_words='english')
pb_matrix  = vectorizer.fit_transform(pb_df['description'])

def retrieve_top_k(query: str, k: int = 5):
    """
    Retrieve the top-K most relevant price book entries for a technician description.
    
    In production: encode query with the same embedding model used for the index,
    then do approximate nearest neighbor search (FAISS) over the index.
    
    Here: TF-IDF cosine similarity (transparent, auditable, fast for small catalogs).
    """
    query_vec  = vectorizer.transform([query])
    sims       = cosine_similarity(query_vec, pb_matrix)[0]
    top_k_idx  = sims.argsort()[-k:][::-1]
    
    results = pb_df.iloc[top_k_idx].copy()
    results['similarity'] = sims[top_k_idx].round(4)
    return results[['code','category','description','labor','parts','total_price','similarity']]

# Test retrieval with a few technician descriptions
test_queries = [
    "replaced the capacitor and checked refrigerant levels, AC is working now",
    "clogged sink, ran the snake about 30 feet, cleared the blockage",
    "igniter was cracked, replaced it and cleaned the flame sensor while I was in there",
]

for q in test_queries:
    print(f"\nQuery: '{q[:70]}...'")
    results = retrieve_top_k(q, k=3)
    print(results[['code','description','total_price','similarity']].to_string(index=False))



Query: 'replaced the capacitor and checked refrigerant levels, AC is working n...'
    code                                                           description  total_price  similarity
HVAC-004                Run capacitor replacement, dual run capacitor 35/5 MFD          105      0.2846
HVAC-001 AC system diagnostic and inspection, includes refrigerant level check           75      0.2704
HVAC-002                                Refrigerant recharge R-410A, per pound           65      0.1938

Query: 'clogged sink, ran the snake about 30 feet, cleared the blockage...'
    code                                                description  total_price  similarity
PLMB-001     Drain snaking, kitchen or bathroom sink, up to 50 feet          120      0.2859
HVAC-005                       Contactor replacement, 2-pole 30 amp           95      0.1793
HVAC-007 AC condensate drain line flush and clean, prevent blockage           75      0.1325

Query: 'igniter was cracked, replaced it and clean

In [4]:
# ─── LLM Estimate Generation (Prompt Design) ─────────────────────────────────
# This shows the full production prompt structure.
# The mock_llm_response simulates what the LLM would return.
# In production: call Claude or GPT-4 with structured output enabled.

SYSTEM_PROMPT = """You are a field service estimating assistant for a home service company.
A technician has described the work they performed. Your job is to:
1. Select the appropriate line items from the provided price book candidates
2. Specify quantities for each line item  
3. Generate a brief customer-facing summary of the work

Respond ONLY with a JSON object in exactly this format:
{
  "line_items": [
    {"code": "HVAC-004", "description": "...", "quantity": 1, "unit_price": 105, "total": 105}
  ],
  "subtotal": 0,
  "customer_summary": "...",
  "technician_notes": "..."
}
Do not include any text outside the JSON object.
"""

def build_estimate_prompt(tech_description: str, candidates: pd.DataFrame) -> str:
    """Build the user prompt combining tech description + retrieved candidates."""
    candidates_text = "\n".join([
        f"- Code: {row['code']} | Description: {row['description']} | "
        f"Labor: ${row['labor']} | Parts: ${row['parts']} | Total: ${row['total_price']}"
        for _, row in candidates.iterrows()
    ])
    
    return f"""Technician description:
\"{tech_description}\"

Available price book candidates (retrieved by relevance):
{candidates_text}

Please select the appropriate line items and generate the estimate JSON.
Only use codes from the candidates list above.
"""

def mock_llm_response(tech_description: str, candidates: pd.DataFrame) -> dict:
    """
    Mock LLM response for demonstration.
    
    In production:
        import anthropic
        client = anthropic.Anthropic()
        response = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            messages=[{
                'role': 'user',
                'content': build_estimate_prompt(tech_description, candidates)
            }]
        )
        return json.loads(response.content[0].text)
    """
    # Simulate: pick the top-2 candidates by similarity as selected items
    selected = candidates.head(2)
    line_items = []
    for _, row in selected.iterrows():
        line_items.append({
            "code": row['code'],
            "description": row['description'],
            "quantity": 1,
            "unit_price": int(row['total_price']),
            "total": int(row['total_price'])
        })
    
    subtotal = sum(item['total'] for item in line_items)
    
    return {
        "line_items": line_items,
        "subtotal": subtotal,
        "customer_summary": (
            f"Our technician performed a diagnostic and repaired your system. "
            f"All work is covered by our 1-year parts and labor warranty."
        ),
        "technician_notes": f"Work performed as described: {tech_description[:100]}"
    }

# ─── Full Pipeline Demo ───────────────────────────────────────────────────────
print("=" * 60)
print("DEMO: RAG Estimating Assistant")
print("=" * 60)

tech_desc = ("AC stopped cooling. Found weak dual run capacitor (35/5 MFD) "
             "and slightly low refrigerant. Replaced capacitor and added 1.5 lbs R-410A.")

print(f"\nTechnician description:\n  {tech_desc}")
print("\n[Step 1] Retrieving relevant price book entries...")
candidates = retrieve_top_k(tech_desc, k=5)
print(candidates[['code','description','total_price','similarity']].to_string(index=False))

print("\n[Step 2] Generating estimate with LLM...")
estimate = mock_llm_response(tech_desc, candidates)

print("\n[Step 3] Structured estimate output:")
print(json.dumps(estimate, indent=2))
print(f"\nEstimate total: ${estimate['subtotal']}")


DEMO: RAG Estimating Assistant

Technician description:
  AC stopped cooling. Found weak dual run capacitor (35/5 MFD) and slightly low refrigerant. Replaced capacitor and added 1.5 lbs R-410A.

[Step 1] Retrieving relevant price book entries...
    code                                                           description  total_price  similarity
HVAC-004                Run capacitor replacement, dual run capacitor 35/5 MFD          105      0.8196
HVAC-002                                Refrigerant recharge R-410A, per pound           65      0.1807
HVAC-001 AC system diagnostic and inspection, includes refrigerant level check           75      0.1112
HVAC-006                  AC evaporator coil cleaning, indoor air handler unit          120      0.0475
HVAC-007            AC condensate drain line flush and clean, prevent blockage           75      0.0457

[Step 2] Generating estimate with LLM...

[Step 3] Structured estimate output:
{
  "line_items": [
    {
      "code": "HVAC-004"

In [5]:
# ─── Evaluation: Retrieval Quality ───────────────────────────────────────────
# In production, evaluate retrieval precision using annotated query-code pairs.
# Here we simulate a small evaluation set.

eval_pairs = [
    ("capacitor replacement on AC unit",    "HVAC-004"),
    ("condenser fan motor not spinning",    "HVAC-003"),
    ("drain backed up in kitchen sink",     "PLMB-001"),
    ("furnace won't ignite, igniter broken","HVAC-102"),
    ("water heater replacement 40 gallon",  "PLMB-002"),
    ("thermostat not working",              "HVAC-008"),
    ("flame sensor dirty furnace",          "HVAC-103"),
    ("GFCI outlet keep tripping",           "ELEC-001"),
]

hits_at_1 = 0
hits_at_3 = 0
hits_at_5 = 0

for query, expected_code in eval_pairs:
    results = retrieve_top_k(query, k=5)
    retrieved_codes = results['code'].tolist()
    
    if expected_code in retrieved_codes[:1]: hits_at_1 += 1
    if expected_code in retrieved_codes[:3]: hits_at_3 += 1
    if expected_code in retrieved_codes[:5]: hits_at_5 += 1
    
    status = "✓" if expected_code in retrieved_codes[:3] else "✗"
    print(f"{status} '{query[:50]}...' → expected {expected_code}, got {retrieved_codes[:3]}")

n = len(eval_pairs)
print(f"\nRetrieval Quality:")
print(f"  Hit@1: {hits_at_1}/{n} = {hits_at_1/n:.0%}")
print(f"  Hit@3: {hits_at_3}/{n} = {hits_at_3/n:.0%}")
print(f"  Hit@5: {hits_at_5}/{n} = {hits_at_5/n:.0%}")
print("\nNote: TF-IDF works surprisingly well for structured domains.")
print("Dense embeddings (text-embedding-3-small) would further improve Hit@1.")


✓ 'capacitor replacement on AC unit...' → expected HVAC-004, got ['HVAC-004', 'HVAC-006', 'HVAC-003']
✓ 'condenser fan motor not spinning...' → expected HVAC-003, got ['HVAC-003', 'HVAC-104', 'HVAC-009']
✓ 'drain backed up in kitchen sink...' → expected PLMB-001, got ['PLMB-001', 'ELEC-001', 'HVAC-007']
✓ 'furnace won't ignite, igniter broken...' → expected HVAC-102, got ['HVAC-102', 'HVAC-105', 'HVAC-104']
✓ 'water heater replacement 40 gallon...' → expected PLMB-002, got ['PLMB-002', 'PLMB-005', 'HVAC-105']
✓ 'thermostat not working...' → expected HVAC-008, got ['HVAC-008', 'SVC-002', 'SVC-003']
✓ 'flame sensor dirty furnace...' → expected HVAC-103, got ['HVAC-103', 'HVAC-105', 'HVAC-104']
✓ 'GFCI outlet keep tripping...' → expected ELEC-001, got ['ELEC-001', 'SVC-002', 'SVC-003']

Retrieval Quality:
  Hit@1: 8/8 = 100%
  Hit@3: 8/8 = 100%
  Hit@5: 8/8 = 100%

Note: TF-IDF works surprisingly well for structured domains.
Dense embeddings (text-embedding-3-small) would further improve 

## Key Takeaways for Interview

1. **RAG is the right architecture for price book lookup** — pure LLM hallucination risk is too high for price-critical fields
2. **TF-IDF retrieval works well for small, structured catalogs** — dense embeddings give better results at scale but add infrastructure complexity
3. **Structured output (JSON schema)** prevents malformed LLM responses — always validate every field before populating the estimate form
4. **Tech review is mandatory** — AI drafts, human approves. Log all edits to improve the model.
5. **Evaluate retrieval separately from generation** — a bad retrieval step is unrecoverable; monitor Hit@3 as the primary retrieval KPI
6. **Fine-tuning opportunity**: collect technician-corrected estimates as training data, fine-tune a smaller model (Llama 3 8B) for 100x cheaper inference at comparable quality
7. **The data flywheel**: tech edits → fine-tuning → better first drafts → fewer edits → faster estimates → more tech adoption → more data
